# Upload files to S3 and check PII (InstantEvidence)

This notebook helps you **put real files into an S3 bucket** and then **see how InstantEvidence (GuardrailStudio) classifies them** for personally identifiable information (PII).

You do not need the AWS CLI or a local traffic generator. You pick files on your computer, the notebook uploads them to S3, and InstantEvidence’s MCP tools report inventory and PII status.

## What you will do (in order)

1. **Install** Python packages used behind the scenes.
2. **Load helpers** — small scripts that talk to AWS and to InstantEvidence.
3. **Check credentials** — confirm AWS and InstantEvidence secrets work.
4. **Upload** files from your laptop into the Colab runtime.
5. **Copy to S3** — send those files into your monitored bucket.
6. **List the bucket** — confirm objects appear in S3 (live AWS view).
7. **PII + inventory** — ask InstantEvidence which objects are PII vs non-PII (governed view).
8. **Cleanup** (optional) — close connections if you are done.

## What you need before starting

- A Google account and [Google Colab](https://colab.research.google.com/).
- An **S3 bucket** that InstantEvidence already monitors (same bucket you use in the product).
- **AWS access keys** limited to that bucket (upload + list; not production admin).
- An **InstantEvidence API key** (`gks_live_…`) and your tenant’s **MCP URL** from the dashboard.

## How to add secrets in Colab

1. In Colab, click the **key icon** in the left sidebar (**Secrets**).
2. Add each name **exactly** as shown (case-sensitive).
3. Toggle **Notebook access** on for each secret you add.

| Secret name | Required | What to put there |
|-------------|----------|-------------------|
| `AWS_ACCESS_KEY_ID` | Yes | Access key for your test IAM user |
| `AWS_SECRET_ACCESS_KEY` | Yes | Secret key for that user |
| `AWS_REGION` | No | Region of the bucket (default `us-east-1` if omitted) |
| `GUARDRAILSTUDIO_MCP_URL` | Yes | MCP endpoint from **Settings → MCP** (ends with `/v1/mcp`) |
| `GUARDRAILSTUDIO_MCP_TOKEN` | Yes | API key from **Settings → API keys** (`gks_live_…`) |

**Where to find InstantEvidence values:** open your InstantEvidence / GuardrailStudio app → **Settings** → copy the MCP URL and create or copy an API key.

## PII status values (what they mean)

After upload, scanning is **not instant**. S3 events must reach InstantEvidence and the DLP worker must finish.

| Status | Meaning |
|--------|--------|
| `NO_PII` | Scanned; no PII detected |
| `HAS_PII` | Scanned; PII detected |
| `PROCESSING` | Scan in progress — wait and re-run the PII cell |
| `UNKNOWN` | Not scanned yet or inconclusive — check bucket wiring and retry |

## Run order

Use **Runtime → Run all**, or run cells top to bottom. The **Load helpers** cell must print `Helpers loaded OK` before anything else that touches AWS or InstantEvidence.

## Step 1 — Install packages

Installs the MCP libraries used to call AWS S3 and the InstantEvidence gateway. This usually takes under a minute. You only need to re-run this cell if Colab shows an import error after a fresh runtime.

In [ ]:
%pip install -q awslabs.aws-api-mcp-server mcp nest_asyncio httpx
print("Packages installed. Continue to Load helpers.")


## Step 2 — Load helpers

Downloads or finds the Python helper files this notebook uses. If you opened the notebook from the [traffic-generator GitHub repo](https://github.com/synapse6-ai/traffic-generator), it uses your local `colab/` folder. Otherwise it downloads the latest helpers from GitHub into `/content/colab-modules`.

**Success:** you see `Helpers loaded OK`.

**If it fails:** use **Runtime → Restart session**, then run this cell again. Advanced: set environment variable `COLAB_HELPERS_RAW_BASE` to a branch-specific raw URL if you are testing unreleased code.

In [ ]:
import os
import sys
import urllib.request
from pathlib import Path

_HELPERS_DIR = Path("/content/colab-modules")
_RAW = os.getenv(
    "COLAB_HELPERS_RAW_BASE",
    "https://raw.githubusercontent.com/synapse6-ai/traffic-generator/main/colab",
).rstrip("/")

if not (_HELPERS_DIR / "colab_bootstrap.py").exists():
    print("First run: downloading bootstrap helper ...")
    _HELPERS_DIR.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f"{_RAW}/colab_bootstrap.py", _HELPERS_DIR / "colab_bootstrap.py")

if str(_HELPERS_DIR) not in sys.path:
    sys.path.insert(0, str(_HELPERS_DIR))

from colab_bootstrap import load_modules

load_modules()

from aws_credentials import CredentialError, mask_access_key, resolve_aws_credentials
from colab_bootstrap import close_mcp_sessions, get_aws_mcp, get_guardrail_mcp
from guardrail_mcp_client import GuardrailCredentialError, GuardrailMcpError, mask_token, resolve_guardrail_credentials
from mcp_s3_client import run_notebook_async

print("Helpers loaded OK — you can run the credentials cell next.")


## Step 3 — Check credentials

Reads your Colab secrets and verifies:

- **AWS** — keys are present (masked in the output).
- **InstantEvidence MCP** — URL and API key are present, and the gateway recognizes your tenant (`auth.whoami`).

**Success:** lines like `AWS OK`, `MCP URL: …`, and `GuardrailStudio OK — tenant=…`.

**Common fixes:**
- Secret name typo (must match the table in the introduction exactly).
- Forgot to enable **Notebook access** on a secret.
- Wrong MCP URL (must include `/v1/mcp`).
- Expired or revoked `gks_live_` API key — create a new one in Settings.

In [ ]:
try:
    aws_creds = resolve_aws_credentials()
    print(f"AWS OK — using access key {mask_access_key(aws_creds.access_key_id)} in region {aws_creds.region}")
except CredentialError as exc:
    raise SystemExit(f"AWS credentials missing:\n{exc}") from exc

try:
    gs_creds = resolve_guardrail_credentials()
    print(f"InstantEvidence MCP URL: {gs_creds.url}")
    print(f"API key: {mask_token(gs_creds.token)} (hidden)")
except GuardrailCredentialError as exc:
    raise SystemExit(f"InstantEvidence MCP credentials missing:\n{exc}") from exc


async def _verify_gs() -> None:
    client = await get_guardrail_mcp(gs_creds)
    whoami = await client.whoami()
    print(
        f"GuardrailStudio OK — signed in as tenant {whoami.get('tenantId')} "
        f"(user {whoami.get('uid')})"
    )


print("Connecting to InstantEvidence MCP (first time may take ~30s) ...")
run_notebook_async(_verify_gs())
print("Credentials look good. Continue to upload files.")


## Step 4 — Upload files from your computer

Click **Choose Files** when you run the cell below. Pick one or more files from your machine (PDF, CSV, text, etc.). They are copied into Colab’s temporary disk at `/content/uploads/` — **not** to S3 yet.

**Success:** each filename and size is printed.

**Tip:** use small test files first so upload and PII scans finish quickly.

In [ ]:
from google.colab import files

UPLOAD_DIR = Path("/content/uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("Click 'Choose Files' and select files from your computer:")
uploaded = files.upload()
local_files: list[Path] = []
for name, data in uploaded.items():
    dest = UPLOAD_DIR / Path(name).name
    dest.write_bytes(data)
    local_files.append(dest)
    print(f"  Ready in Colab: {dest.name} ({dest.stat().st_size:,} bytes)")

if not local_files:
    print("No files selected — run this cell again and pick at least one file.")
else:
    print(f"{len(local_files)} file(s) ready. Next: Transfer to S3.")


## Step 5 — Transfer to S3

Uploads each file from Colab into your S3 bucket using the AWS MCP integration.

**Set these parameters:**

- **BUCKET** — must be the **same bucket** InstantEvidence monitors (the one wired to S3 event notifications).
- **KEY_PREFIX** — folder prefix inside the bucket (e.g. `uploads/`). Objects will be stored as `uploads/yourfile.pdf`.

**Success:** each line shows `OK` and an `s3://bucket/key` path.

**If upload fails:** check bucket name, region secret, and IAM permissions (`s3:PutObject`, `s3:HeadBucket` on that bucket).

In [ ]:
import mimetypes

BUCKET = "your-instantevidence-bucket"  # @param {type:"string"}
KEY_PREFIX = "uploads/"  # @param {type:"string"}

if not local_files:
    raise SystemExit(
        "No files to upload. Run Step 4 (Upload files) and select at least one file."
    )

prefix = KEY_PREFIX.strip()
if prefix and not prefix.endswith("/"):
    prefix += "/"

print(f"Uploading {len(local_files)} file(s) to s3://{BUCKET}/ ...")


async def _upload_all() -> list[dict]:
    mcp = await get_aws_mcp(aws_creds)
    head = await mcp.head_bucket(BUCKET)
    if not head.ok:
        raise RuntimeError(head.error or "Cannot access bucket — check name, region, and IAM")
    results: list[dict] = []
    for path in local_files:
        key = f"{prefix}{path.name}" if prefix else path.name
        ctype, _ = mimetypes.guess_type(path.name)
        out = await mcp.put_object_file(
            BUCKET, key, path, content_type=ctype or "application/octet-stream"
        )
        results.append({"file": path.name, "key": key, "ok": out.ok, "error": out.error})
        status = "OK" if out.ok else out.error
        print(f"  {path.name} → s3://{BUCKET}/{key} [{status}]")
    return results


upload_results = run_notebook_async(_upload_all())
ok = sum(1 for r in upload_results if r["ok"])
print(f"Done: {ok}/{len(upload_results)} uploaded to S3.")
if ok:
    print("Next: List bucket to confirm objects, then run PII + inventory in InstantEvidence.")


## Step 6 — List bucket (live S3 view)

Lists objects currently in the bucket under **LIST_PREFIX** (defaults to the same prefix you used for upload). This reads **directly from AWS**, not from InstantEvidence’s database.

**Success:** you see the keys you just uploaded with sizes.

**If the list is empty:** wrong bucket, wrong prefix, or upload step failed — fix Step 5 first.

In [ ]:
LIST_PREFIX = KEY_PREFIX  # @param {type:"string"}
MAX_KEYS = 100  # @param {type:"integer"}


def _format_bytes(n: int | None) -> str:
    if n is None:
        return "-"
    size = float(n)
    for unit in ("B", "KB", "MB", "GB"):
        if size < 1024:
            return f"{size:.0f} {unit}" if unit == "B" else f"{size:.1f} {unit}"
        size /= 1024
    return f"{size:.1f} TB"


print(f"Listing s3://{BUCKET}/ (prefix={LIST_PREFIX!r}, up to {MAX_KEYS} keys) ...")


async def _list_bucket() -> list[dict]:
    mcp = await get_aws_mcp(aws_creds)
    out = await mcp.list_objects(BUCKET, prefix=LIST_PREFIX, max_keys=MAX_KEYS)
    if not out.ok:
        raise RuntimeError(out.error or "list-objects failed")
    rows: list[dict] = []
    contents = (out.data or {}).get("Contents") or []
    if not contents:
        print("  (no objects found — check BUCKET and LIST_PREFIX)")
    for obj in contents:
        rows.append({"Key": obj.get("Key"), "Size": _format_bytes(obj.get("Size"))})
        print(f"  {obj.get('Key')}  ({_format_bytes(obj.get('Size'))})")
    return rows


bucket_objects = run_notebook_async(_list_bucket())
print(f"Found {len(bucket_objects)} object(s) in S3.")


## Step 7 — PII classification and governed inventory (InstantEvidence)

This step uses **InstantEvidence’s MCP gateway** (not raw S3). It runs two queries:

1. **PII (`nl.query`)** — natural-language question over your tenant data. Returns which objects are `HAS_PII`, `NO_PII`, `PROCESSING`, etc. **Can take several minutes.** Output format may vary; read the JSON printed below.
2. **Inventory (`object.metadata.search`)** — governed object list from InstantEvidence (bucket, key, size). Only includes objects InstantEvidence knows about after onboarding and scans.

**PROJECT_NAME** (optional) — if you have multiple projects in InstantEvidence, enter the **display name** (or substring) of the project tied to this bucket. Leave blank to search across projects you can access.

**After a fresh upload:** expect `PROCESSING` or `UNKNOWN` until S3 events reach InstantEvidence and DLP finishes. Wait a few minutes and **re-run this cell**.

**Success:** JSON under “PII” and a list of `bucket/key` lines under “Inventory”.

**In the InstantEvidence UI:** you should also see new S3 events and receipts for the same bucket if event delivery (SNS / EventBridge) is configured.

In [ ]:
import json

PROJECT_NAME = ""  # @param {type:"string"}

pii_question = (
    f"List governed S3 objects in bucket {BUCKET!r} with key prefix {LIST_PREFIX.strip()!r}. "
    "Include bucket, key, and pii_status (HAS_PII, NO_PII, PROCESSING, UNKNOWN, NOT_TRACKED). "
    "Limit 100 rows."
)
proj = PROJECT_NAME.strip() or None
if proj:
    print(f"Scoping queries to project name matching: {proj!r}")
else:
    print("No PROJECT_NAME set — querying across accessible projects.")


async def _guardrail_queries() -> tuple[object, dict]:
    client = await get_guardrail_mcp(gs_creds)
    print("Running PII query (nl.query) — please wait ...")
    pii = await client.nl_query(pii_question, project_name=proj)
    print("Running governed inventory (object.metadata.search) ...")
    inventory = await client.object_metadata_search(
        project_name=proj,
        key_prefix=LIST_PREFIX.strip() or None,
        limit=100,
    )
    return pii, inventory


try:
    pii_result, inventory = run_notebook_async(_guardrail_queries())
except GuardrailMcpError as exc:
    raise SystemExit(f"InstantEvidence MCP error:\n{exc}") from exc

print("\n=== PII results (from nl.query) ===")
print(
    json.dumps(pii_result, indent=2, default=str)[:8000]
    if isinstance(pii_result, (dict, list))
    else pii_result
)
print("\n=== Governed inventory (from object.metadata.search) ===")
items = inventory.get("items") or []
if not items:
    print("  (no governed objects yet — confirm bucket is onboarded in InstantEvidence)")
for item in items:
    print(f"  {item.get('bucketName')}/{item.get('key')}")
print(f"Total in inventory: {inventory.get('total', len(items))}")
print("\nIf PII status is still PROCESSING, wait and re-run this cell.")


## Step 8 — Cleanup (optional)

Closes the background connections to AWS MCP and InstantEvidence MCP. Run this if you are **finished** with the notebook.

Skip this cell if you plan to re-run upload, list, or PII cells in the same Colab session — those cells reuse the same connections.

In [ ]:
run_notebook_async(close_mcp_sessions())
print("Connections closed. To run S3 or InstantEvidence steps again, restart from Step 3 (credentials).")


## Troubleshooting

| Problem | What to try |
|---------|-------------|
| `Helpers loaded OK` never appears | **Runtime → Restart session**, run Step 1–2 again |
| AWS credentials error | Check secret names; enable Notebook access on secrets |
| `head-bucket failed` | Wrong **BUCKET** or **AWS_REGION**; IAM needs `s3:HeadBucket` |
| Upload `AccessDenied` | IAM needs `s3:PutObject` on `arn:aws:s3:::BUCKET/*` |
| List empty but upload OK | **LIST_PREFIX** must match **KEY_PREFIX** (e.g. both `uploads/`) |
| MCP auth / 401 | Regenerate `gks_live_` key; verify **GUARDRAILSTUDIO_MCP_URL** ends with `/v1/mcp` |
| PII always `PROCESSING` | Wait 2–5 min; confirm S3 events reach InstantEvidence; re-run Step 7 |
| Inventory empty | Onboard the bucket in InstantEvidence; run a bucket scan in the product |
| `nl.query` timeout | Normal for large tenants; retry; narrow with **PROJECT_NAME** |

Repository: [synapse6-ai/traffic-generator](https://github.com/synapse6-ai/traffic-generator) — report issues there.